# UHI Data Extraction Pipeline

This notebook extracts all satellite data from scratch and produces the 5 datasets used for UHI classification. Every function is self-contained here, no external package imports required. The pipeline queries Microsoft Planetary Computer (free, no authentication) for Sentinel-2, Landsat-8, and Copernicus DEM data, computes spectral indices and Land Surface Temperature, and attaches 3D building morphology features from the 3D-GloBFP dataset (Li et al., 2024).

The 5 output files are: `dataset_100m_Chile.csv`, `dataset_100m_Brazil.csv`, `dataset_100m_Sierra_Leone.csv` (spectral + thermal + elevation), plus `buildings_Chile.csv` and `buildings_Brazil.csv` (same data enriched with 15 building morphology features). Sierra Leone has no building data because 3D-GloBFP coverage is incomplete for Freetown.

Li, Y., Xu, Y., Qiu, D., Wang, J., Li, H., & Wu, B. (2024). 3D-GloBFP: The first global three-dimensional building footprint dataset. *Earth System Science Data*, *16*(11), 5357-5374. https://doi.org/10.5194/essd-16-5357-2024

In [1]:
# Run once if needed. These are the satellite data access libraries.
# pip install pystac-client planetary-computer odc-stac rioxarray rasterio xarray geopandas shapely scipy tqdm

In [ ]:
import os, time, zipfile, warnings, requests
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rio
import rasterio
import rasterio.transform
import requests
from rasterio.merge import merge as rio_merge
from odc.stac import stac_load
import pystac_client
import planetary_computer
import geopandas as gpd
from shapely.geometry import Point, box
from scipy.spatial import cKDTree

warnings.filterwarnings('ignore')

OUT_DIR = r"C:\Users\micki\OneDrive\Desktop\Coding\BC-II\Final"
os.makedirs(OUT_DIR, exist_ok=True)

## Location Configurations

Each city has a point CSV (from the competition GitHub repo), a geographic bounding box that defines the satellite download area, and a time window for scene search. The time windows were chosen to capture peak summer conditions when the UHI signal is strongest. Chile uses January 2024 (Southern Hemisphere summer), Brazil uses January to March 2023 (tropical wet season with clear skies between storms), and Sierra Leone uses January to February 2023 (dry season with minimal cloud cover).

The bounding boxes extend beyond the point grid to include the surrounding geography that influences local microclimates. For Brazil, the bbox reaches into the Atlantic coastline and the forested Serra do Mar foothills; for Sierra Leone, it captures the ocean fringe of the Freetown Peninsula and the elevated interior ridgeline. This coverage ensures the neighborhood extraction at boundary points draws from physically complete pixel windows rather than clipping against the raster edge, and it preserves the full thermal gradient between coastal cooling and inland heat accumulation that the model later relies on to separate UHI classes.

In [3]:
CHILE = {
    'label': 'Chile',
    'points': 'https://raw.githubusercontent.com/chase-kusterer/bc-II/main/Data/sample_chile_uhi_data.csv',
    'bbox': (-33.88, -71.05, -33.23, -70.04),  # (south, west, north, east)
    'time_window': '2024-01-15/2024-02-15',
}

BRAZIL = {
    'label': 'Brazil',
    'points': 'https://raw.githubusercontent.com/chase-kusterer/bc-II/main/Data/Sample_Brazil_uhi_data.csv',
    'bbox': (-23.015, -43.52, -22.82, -43.17),
    'time_window': '2023-01-01/2023-03-15',
}

SIERRA_LEONE = {
    'label': 'Sierra Leone',
    'points': 'https://raw.githubusercontent.com/chase-kusterer/bc-II/main/Data/Validation_Dataset.csv',
    'bbox': (8.36, -13.32, 8.52, -13.10),
    'time_window': '2023-01-15/2023-02-28',
}

LOCATIONS = [CHILE, BRAZIL, SIERRA_LEONE]

## Helper Functions

`load_points` normalizes column names from the competition CSVs. The STAC client connects to Microsoft Planetary Computer, which hosts all three satellite collections we need (Sentinel-2 L2A, Landsat Collection 2 Level 2, and Copernicus DEM GLO-30). No API key or authentication is required.

In [4]:
def load_points(source):
    """Load point locations from CSV path, URL, or DataFrame. Normalizes Lat/Lon column casing."""
    if isinstance(source, pd.DataFrame):
        df = source.copy()
    else:
        df = pd.read_csv(source)
    col_map = {}
    for c in df.columns:
        if c.lower() == 'latitude': col_map[c] = 'Latitude'
        elif c.lower() == 'longitude': col_map[c] = 'Longitude'
    df.rename(columns=col_map, inplace=True)
    return df

def bbox_to_stac(bbox):
    """Convert (south, west, north, east) to STAC format (west, south, east, north)."""
    south, west, north, east = bbox
    return (west, south, east, north)

def stac_client():
    """Connect to Microsoft Planetary Computer STAC catalog."""
    return pystac_client.Client.open('https://planetarycomputer.microsoft.com/api/stac/v1')

## Pixel Extraction Strategy

A 5x5 neighborhood at 100m resolution produces a 500m x 500m ground footprint per sample point. We tested sizes from 0 (nearest pixel) through 10 and found 5 optimal, it reduced LST variance without losing spatial detail, while larger windows began blurring class boundaries at urban-vegetation edges. At this scale, each pixel window captures the proportional mix of rooftops, streets, courtyards, and tree canopy that collectively determines how heat is absorbed and released rather than the spectral signature of whichever single surface the coordinate lands on. This is closer to what UHI classification actually targets: the aggregate thermal character of a neighborhood, not the temperature of any one object within it.

In [5]:
def extract_at_points(comp, bands, pts_df, neighborhood, scale):
    """
    Extract band values at point locations from a composited raster.
    neighborhood=0: single nearest pixel. neighborhood=N: NxN pixel bounding box median.
    """
    lats = pts_df['Latitude'].values
    lons = pts_df['Longitude'].values
    df = pd.DataFrame({'Latitude': lats, 'Longitude': lons})

    if neighborhood <= 0:
        lat_da = xr.DataArray(lats, dims='points')
        lon_da = xr.DataArray(lons, dims='points')
        for band in bands:
            df[band] = comp[band].sel(longitude=lon_da, latitude=lat_da, method='nearest').values.astype(float)
    else:
        half_pixels = neighborhood / 2
        lon_offset = scale * half_pixels
        lat_offset = scale * half_pixels
        for band in bands:
            vals = []
            for lat, lon in zip(lats, lons):
                subset = comp[band].sel(
                    longitude=slice(lon - lon_offset, lon + lon_offset),
                    latitude=slice(lat + lat_offset, lat - lat_offset),
                )
                vals.append(float(np.nanmedian(subset.values)))
            df[band] = vals
    return df

## Sentinel-2 L2A Extraction

Sentinel-2 provides 13 spectral bands from 10 to 60m resolution covering visible through shortwave infrared. We use the Scene Classification Layer (SCL, Band 60) for pixel-level cloud masking rather than filtering entire scenes by cloud percentage. This is more aggressive but yields zero NaN across all three cities because we keep every clear pixel from partially cloudy scenes that would otherwise be discarded entirely.

After masking clouds, shadows, cirrus, snow, and saturated pixels, we compute a median composite across all available scenes in the time window. Median is chosen over mean because it is robust to outlier pixels that slip through the cloud mask. The result is a single clean raster representing typical surface reflectance during the study period.

We extract 11 bands (B01 through B12, excluding B09 and B10 which are atmospheric/cirrus bands not useful for surface analysis).

In [6]:
SENTINEL_BANDS = ['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B11', 'B12']
SCL_BAD = {0, 1, 3, 8, 9, 10}  # nodata, saturated, shadow, cloud-medium, cloud-high, cirrus

def extract_sentinel(pts_df, bbox, time_window, resolution=100, max_cloud=30, neighborhood=5):
    """Extract Sentinel-2 bands at each point. SCL cloud mask + median composite."""
    stac_bbox = bbox_to_stac(bbox)
    scale = resolution / 111320.0

    stac = stac_client()
    search = stac.search(
        bbox=stac_bbox, datetime=time_window,
        collections=['sentinel-2-l2a'],
        query={'eo:cloud_cover': {'lt': max_cloud}},
    )
    items = list(search.get_items())
    print(f'[Sentinel-2] {len(items)} scenes | res={resolution}m | {time_window}')

    load_bands = SENTINEL_BANDS + ['SCL']
    data = stac_load(
        items, bands=load_bands, crs='EPSG:4326', resolution=scale,
        chunks={'x': 2048, 'y': 2048}, dtype='uint16',
        patch_url=planetary_computer.sign, bbox=stac_bbox,
    )

    # SCL cloud mask: keep only clear pixels
    scl = data['SCL']
    cloud_mask = scl.copy(data=np.ones_like(scl.values, dtype=bool))
    for bad_val in SCL_BAD:
        cloud_mask = cloud_mask & (scl != bad_val)

    data_clean = data[SENTINEL_BANDS].where(cloud_mask)
    data_clean = data_clean.where(data_clean > 0)  # mask raw nodata (DN=0)
    comp = data_clean.median(dim='time').compute()

    df = extract_at_points(comp, SENTINEL_BANDS, pts_df, neighborhood, scale)
    del data, data_clean, comp

    valid = df[SENTINEL_BANDS].notna().all(axis=1).sum()
    print(f'[Sentinel-2] {len(df)} points x {len(SENTINEL_BANDS)} bands ({valid} valid)')
    return df

## Landsat-8 Extraction

Landsat-8 provides the thermal infrared band (lwir11, Band 10) that we need for Land Surface Temperature, plus visible/NIR bands at 30m. The thermal band has 100m native resolution. We originally used a single scene with no cloud masking, which caused catastrophic failure in Brazil: the first scene was 47% cloud-covered, producing -60C brightness temperatures where clouds appeared. The fix applies QA_PIXEL bit masking (dilated cloud, cloud, cloud shadow, snow, fill) and composites across all available scenes.

Landsat surface reflectance bands require USGS Collection 2 Level 2 scale factors (multiply by 0.0000275, offset by -0.2). The thermal band converts from Digital Number to brightness temperature in Celsius using a separate scale (multiply by 0.00341802, add 149.0 for Kelvin, then subtract 273.15). Brightness temperature assumes a perfect blackbody emissivity of 1.0. The actual LST correction using NDVI-derived emissivity is applied later in the feature engineering step.

In [7]:
LANDSAT_VIS = ['red', 'green', 'blue', 'nir08']
LANDSAT_THERMAL = ['lwir11']
QA_CLOUD_BITS = [1, 3, 4, 5]  # dilated cloud, cloud, cloud shadow, snow

# USGS Collection 2 Level 2 scale factors
VIS_SCALE, VIS_OFFSET = 0.0000275, -0.2
THERM_SCALE, THERM_OFFSET = 0.00341802, 149.0  # converts to Kelvin

def qa_cloud_mask(qa_band):
    """Build boolean mask from QA_PIXEL. True = clear pixel."""
    mask = np.ones_like(qa_band.values, dtype=bool)
    for bit in QA_CLOUD_BITS:
        mask = mask & ((qa_band.values.astype(int) >> bit) & 1 == 0)
    mask = mask & ((qa_band.values.astype(int) >> 0) & 1 == 0)  # fill bit
    return qa_band.copy(data=mask)

def extract_landsat(pts_df, bbox, time_window, resolution=100, max_cloud=50, neighborhood=5):
    """Extract Landsat-8 vis + thermal at each point. QA_PIXEL mask + median composite."""
    all_bands = LANDSAT_VIS + LANDSAT_THERMAL
    stac_bbox = bbox_to_stac(bbox)
    scale = resolution / 111320.0

    stac = stac_client()
    search = stac.search(
        bbox=stac_bbox, datetime=time_window,
        collections=['landsat-c2-l2'],
        query={'eo:cloud_cover': {'lt': max_cloud}, 'platform': {'in': ['landsat-8']}},
    )
    items = list(search.get_items())
    print(f'[Landsat] {len(items)} scenes | res={resolution}m | {time_window}')
    for i, item in enumerate(items):
        dt = item.properties.get('datetime', '?')
        cc = item.properties.get('eo:cloud_cover', '?')
        print(f'[Landsat]   scene {i}: {dt}  cloud={cc}%')

    # Load vis + QA together, thermal separately (different native resolution)
    data_vis = stac_load(
        items, bands=LANDSAT_VIS + ['qa_pixel'], crs='EPSG:4326', resolution=scale,
        chunks={'x': 2048, 'y': 2048}, dtype='uint16',
        patch_url=planetary_computer.sign, bbox=stac_bbox,
    )
    data_therm = stac_load(
        items, bands=LANDSAT_THERMAL, crs='EPSG:4326', resolution=scale,
        chunks={'x': 2048, 'y': 2048}, dtype='uint16',
        patch_url=planetary_computer.sign, bbox=stac_bbox,
    )

    clear_mask = qa_cloud_mask(data_vis['qa_pixel'])
    data_vis_clean = data_vis[LANDSAT_VIS].where(clear_mask).where(data_vis[LANDSAT_VIS] > 0)
    data_therm_clean = data_therm.where(clear_mask).where(data_therm > 0)

    # Apply USGS scale factors
    vis_scaled = data_vis_clean.astype(float) * VIS_SCALE + VIS_OFFSET
    therm_scaled = data_therm_clean.astype(float) * THERM_SCALE + THERM_OFFSET - 273.15

    comp_vis = vis_scaled.median(dim='time').compute()
    comp_therm = therm_scaled.median(dim='time').compute()
    combined = xr.merge([comp_vis, comp_therm])

    df = extract_at_points(combined, all_bands, pts_df, neighborhood, scale)
    del data_vis, data_therm, combined

    valid = df['lwir11'].notna().sum()
    print(f'[Landsat] {len(df)} points x {len(all_bands)} bands ({valid} valid thermal)')
    return df

## Copernicus DEM GLO-30

Elevation is a static feature, it does not change with time or resolution. We extract it once per city from the Copernicus GLO-30 dataset (30m global coverage). For Santiago, elevation ranges from 388 to 981m and is the single strongest UHI predictor. For Rio and Freetown, both coastal cities, elevation has limited dynamic range but still captures terrain effects on local temperature.

In [8]:
def extract_dem(pts_df, bbox):
    """Extract elevation at each point from Copernicus DEM GLO-30."""
    stac_bbox = bbox_to_stac(bbox)
    stac = pystac_client.Client.open(
        'https://planetarycomputer.microsoft.com/api/stac/v1',
        modifier=planetary_computer.sign_inplace,
    )
    search = stac.search(bbox=stac_bbox, collections=['cop-dem-glo-30'])
    items = list(search.get_items())
    print(f'[DEM] {len(items)} tiles')

    datasets = [rasterio.open(planetary_computer.sign(item.assets['data']).href) for item in items]
    merged_array, merged_transform = rio_merge(datasets)
    merged_array = merged_array[0].astype('float32')
    merged_array[merged_array < -1000] = float('nan')
    for ds in datasets:
        ds.close()

    n_rows, n_cols = merged_array.shape
    west, south, east, north = rasterio.transform.array_bounds(n_rows, n_cols, merged_transform)
    lat_coords = np.linspace(north, south, n_rows)
    lon_coords = np.linspace(west, east, n_cols)

    xr_elev = xr.Dataset({
        'elevation': xr.DataArray(merged_array, dims=['y', 'x'],
                                   coords={'y': lat_coords, 'x': lon_coords})
    })

    s, w, n, e = bbox
    clipped = xr_elev.sel(x=slice(w, e), y=slice(n, s))

    df = pd.DataFrame({
        'Latitude': pts_df['Latitude'].values,
        'Longitude': pts_df['Longitude'].values,
        'elevation': clipped.elevation.sel(
            x=xr.DataArray(pts_df['Longitude'].values, dims='points'),
            y=xr.DataArray(pts_df['Latitude'].values, dims='points'),
            method='nearest'
        ).values.astype(float),
    })

    del merged_array, xr_elev, clipped
    print(f'[DEM] {len(df)} points | range: {df["elevation"].min():.0f} to {df["elevation"].max():.0f}m')
    return df

## Feature Engineering

From the raw Sentinel-2 bands we compute 19 spectral indices that isolate different land cover properties. Vegetation indices (NDVI, EVI, SAVI, GNDVI, LAI) capture canopy cover and evapotranspirative cooling. Water indices (NDWI, MNDWI, NDMI) identify water bodies and soil moisture. Built-up indices (NDBI, ISA, BU, NDISI) measure impervious surfaces. Soil indices (NDBaI, BSI, DBSI) differentiate bare ground from paved areas. Physical surface properties (LSE, Albedo, SWIR1_NIR, SWIR2_NIR) capture the energy balance and reflectivity.

From Landsat, we derive NDVI (separate from Sentinel NDVI), emissivity using Sobrino et al. (2004) thresholds, and Land Surface Temperature via the mono-window method. The mono-window formula corrects brightness temperature by accounting for the fact that real surfaces are not perfect blackbodies. The correction uses NDVI-derived emissivity as a proxy for surface material type: bare soil and water have lower emissivity (0.97), full vegetation has higher (0.99), and mixed pixels are interpolated.

In [9]:
def sentinel_features(df):
    """Compute 19 spectral indices from Sentinel-2 raw DN bands."""
    B02, B03, B04 = df['B02'], df['B03'], df['B04']
    B08, B11, B12 = df['B08'], df['B11'], df['B12']

    # Vegetation
    df['NDVI']  = (B08 - B04) / (B08 + B04)
    df['EVI']   = 2.5 * (B08 - B04) / (B08 + 6*B04 - 7.5*B02 + 10000)
    df['SAVI']  = (B08 - B04) / (B08 + B04 + 0.5) * 1.5
    df['GNDVI'] = (B08 - B03) / (B08 + B03)
    df['LAI']   = 0.57 * np.exp(2.33 * df['NDVI'])

    # Water
    df['NDWI']  = (B03 - B08) / (B03 + B08)
    df['MNDWI'] = (B03 - B11) / (B03 + B11)
    df['NDMI']  = (B08 - B11) / (B08 + B11)

    # Built-up / Urban
    df['NDBI']  = (B11 - B08) / (B11 + B08)
    df['ISA']   = (df['NDBI'] - df['NDVI']) / (df['NDBI'] + df['NDVI'])
    df['BU']    = df['NDBI'] - df['NDVI']
    df['NDISI'] = (B11 - (B02+B08+B12)/3) / (B11 + (B02+B08+B12)/3)

    # Soil / Bare
    df['NDBaI'] = (B11 - B12) / (B11 + B12)
    df['BSI']   = ((B11+B04) - (B08+B02)) / ((B11+B04) + (B08+B02))
    df['DBSI']  = (B11 - B03) / (B11 + B03) - df['NDVI']

    # Physical surface
    Pv = np.clip(((df['NDVI'] - 0.2) / (0.5 - 0.2))**2, 0, 1)
    df['LSE'] = 0.004 * Pv + 0.986
    df['Albedo'] = (0.356*(B02/10000) + 0.130*(B04/10000) + 0.373*(B08/10000)
                    + 0.085*(B11/10000) + 0.072*(B12/10000) - 0.0018)

    # Band ratios
    df['SWIR1_NIR'] = B11 / (B08 + 1e-10)
    df['SWIR2_NIR'] = B12 / (B08 + 1e-10)
    return df

def landsat_features(df):
    """Compute LST from Landsat brightness temperature using mono-window method."""
    df['NDVI_Landsat'] = (df['nir08'] - df['red']) / (df['nir08'] + df['red'])

    # Emissivity from NDVI (Sobrino et al., 2004)
    pv = np.clip(((df['NDVI_Landsat'] - 0.2) / (0.5 - 0.2))**2, 0, 1)
    emissivity = np.where(df['NDVI_Landsat'] < 0.2, 0.97,
                          np.where(df['NDVI_Landsat'] > 0.5, 0.99, 0.004 * pv + 0.986))
    df['emissivity'] = emissivity

    # LST: BT(C) -> BT(K) -> LST(K) -> LST(C)
    BT_kelvin = df['lwir11'] + 273.15
    wavelength = 10.895   # um, Landsat 8 TIRS Band 10
    rho = 14388.0         # um*K, Planck radiation constant
    df['LST'] = BT_kelvin / (1 + (wavelength * BT_kelvin / rho) * np.log(emissivity)) - 273.15
    return df

In [10]:
def merge_sources(dfs):
    """Left-join a list of DataFrames on (Latitude, Longitude)."""
    base = dfs[0].copy()
    for df in dfs[1:]:
        overlap = [c for c in df.columns if c in base.columns and c not in ('Latitude', 'Longitude')]
        right = df.drop(columns=overlap)
        base = base.merge(right, on=['Latitude', 'Longitude'], how='left')
    return base

## Build Dataset

The `build_dataset` function orchestrates the full pipeline for one city: load points, extract Sentinel-2, Landsat, and DEM, compute spectral features, merge on coordinates, and preserve the UHI_Class label from the original competition CSV. The resolution is fixed at 100m, which our resolution sweep experiments confirmed as the optimal tradeoff between spatial detail and noise reduction.

In [11]:
def build_dataset(points, bbox, time_window, label=None, resolution=100, neighborhood=5):
    """Full extraction pipeline for one city. Returns ML-ready DataFrame."""
    pts_df = load_points(points)
    dfs = []

    print(f'\n  Extracting Sentinel-2...')
    df_s = extract_sentinel(pts_df, bbox, time_window, resolution, neighborhood=neighborhood)
    if df_s is not None:
        df_s = sentinel_features(df_s)
        dfs.append(df_s)

    print(f'  Extracting Landsat-8...')
    df_l = extract_landsat(pts_df, bbox, time_window, resolution, neighborhood=neighborhood)
    if df_l is not None:
        df_l = landsat_features(df_l)
        dfs.append(df_l)

    print(f'  Extracting DEM...')
    df_d = extract_dem(pts_df, bbox)
    if df_d is not None:
        dfs.append(df_d)

    merged = merge_sources(dfs)

    # Preserve UHI_Class from original CSV
    if 'UHI_Class' in pts_df.columns and 'UHI_Class' not in merged.columns:
        merged['UHI_Class'] = pts_df['UHI_Class'].values
    if label:
        merged['label'] = label

    print(f'  Result: {merged.shape[0]} rows x {merged.shape[1]} cols')
    return merged

## Extract All Three Cities

This runs the full pipeline sequentially for Chile, Brazil, and Sierra Leone. Each city takes roughly 5 to 15 minutes depending on the number of Sentinel-2 scenes available and network speed. The outputs are saved as the 3 `dataset_100m_*.csv` files.

In [12]:
for loc in LOCATIONS:
    name = loc['label']
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")

    df = build_dataset(
        points=loc['points'],
        bbox=loc['bbox'],
        time_window=loc['time_window'],
        label=name,
    )

    out_file = f'dataset_100m_{name.replace(" ", "_")}.csv'
    out_path = os.path.join(OUT_DIR, out_file)
    df.to_csv(out_path, index=False)
    print(f'  Saved: {out_path}')
    print(f'  LST range: {df["LST"].min():.1f} to {df["LST"].max():.1f} C')
    print(f'  Elevation range: {df["elevation"].min():.0f} to {df["elevation"].max():.0f} m')


  Chile

  Extracting Sentinel-2...
[Sentinel-2] 55 scenes | res=100m | 2024-01-15/2024-02-15
[Sentinel-2] 21662 points x 11 bands (21662 valid)
  Extracting Landsat-8...
[Landsat] 8 scenes | res=100m | 2024-01-15/2024-02-15
[Landsat]   scene 0: 2024-02-15T14:27:57.396371Z  cloud=0.46%
[Landsat]   scene 1: 2024-02-15T14:27:33.441790Z  cloud=0.28%
[Landsat]   scene 2: 2024-02-06T14:34:10.611221Z  cloud=9.28%
[Landsat]   scene 3: 2024-02-06T14:33:46.656641Z  cloud=10.29%
[Landsat]   scene 4: 2024-01-30T14:27:56.562312Z  cloud=1.23%
[Landsat]   scene 5: 2024-01-30T14:27:32.607731Z  cloud=7.01%
[Landsat]   scene 6: 2024-01-21T14:34:09.201547Z  cloud=6.68%
[Landsat]   scene 7: 2024-01-21T14:33:45.246966Z  cloud=9.66%
[Landsat] 21662 points x 5 bands (21662 valid thermal)
  Extracting DEM...
[DEM] 2 tiles
[DEM] 21662 points | range: 388 to 981m
  Result: 21662 rows x 43 cols
  Saved: C:\Users\micki\OneDrive\Desktop\Coding\BC-II\Final\dataset_100m_Chile.csv
  LST range: 32.4 to 51.9 C
  Elev

## Building Morphology Features

The 3D Global Building Footprint dataset (3D-GloBFP) provides building footprints with satellite-derived estimated heights (Li et al., 2024). For each sample point, we compute 15 morphology features within a 100m square buffer: basic counts and density, height statistics (mean, max, std, range), volume metrics (total volume, volume density, mean building volume), coverage ratios (building coverage, open space), and derived UHI proxies like sky view factor and compactness. These features capture the urban canyon effect, where tall, dense buildings trap heat by reducing sky view and ventilation. The building coverage ratio and compactness index are the strongest building-derived UHI predictors, though their signal is weaker than spectral indices (Spearman rho of 0.15 vs 0.4 for LST).

We used tiles grid_250 and grid_252 for Santiago and grid_481 for Rio de Janeiro, freely available on Zenodo at https://zenodo.org/records/15487037. Each shapefile contains individual building polygons with a Height attribute estimated via XGBoost from multisource Earth Observation data. The competition repository (bc-II/Data/Building_Footprint_Data.zip) includes pre-clipped versions of these footprints but without the Height column, so the full morphology extraction requires downloading the original tiles from Zenodo. We build a KDTree spatial index on building centroids for fast radius queries, making the per-point computation roughly O(log n) instead of O(n). Chile and Brazil have building data; Sierra Leone coverage is incomplete so we skip it.

In [15]:
# 3D-GloBFP tile download via Figshare API
# Source: Li et al. (2024), https://doi.org/10.5281/zenodo.11319912
# Each tile is a zip containing a shapefile with building footprints + Height column

FIGSHARE_ARTICLES = [28879733, 28881749]  # Part I (grid 0-400), Part II (grid 400+)

TILE_CONFIG = {
    'Chile': {
        'tiles': ['250_-71.25_-33.75_-70.0_-32.5_AR_CI.zip',
                   '252_-71.25_-35.0_-70.0_-33.75_AR_CI.zip'],
        'bbox': (-71.05, -33.88, -70.04, -33.23),
    },
    'Brazil': {
        'tiles': ['481_-45.0_-25.0_-42.5_-22.5_BR.zip'],
        'bbox': (-43.80, -23.10, -43.10, -22.75),
    },
}

BLDG_DIR = os.path.join(OUT_DIR, '3dglobfp')
os.makedirs(BLDG_DIR, exist_ok=True)

# Build a lookup of filename -> download_url from Figshare
needed = {z for cfg in TILE_CONFIG.values() for z in cfg['tiles']}
url_map = {}
for art_id in FIGSHARE_ARTICLES:
    r = requests.get(f'https://api.figshare.com/v2/articles/{art_id}/files?limit=1000')
    r.raise_for_status()
    for f in r.json():
        if f['name'] in needed:
            url_map[f['name']] = f['download_url']
    if len(url_map) == len(needed):
        break

print(f'Resolved {len(url_map)}/{len(needed)} tile URLs from Figshare API')

def download_tile(zip_name, dest_dir):
    """Download a single 3D-GloBFP tile zip from Figshare and extract it."""
    tile_dir = os.path.join(dest_dir, zip_name.replace('.zip', ''))
    if os.path.exists(tile_dir):
        print(f'  Already extracted: {tile_dir}')
        return tile_dir

    url = url_map[zip_name]
    local_zip = os.path.join(dest_dir, zip_name)
    print(f'  Downloading {zip_name}...')
    r = requests.get(url, stream=True)
    r.raise_for_status()
    with open(local_zip, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f'  Downloaded {os.path.getsize(local_zip) / 1e6:.1f} MB')

    print(f'  Extracting...')
    with zipfile.ZipFile(local_zip, 'r') as z:
        z.extractall(tile_dir)
    os.remove(local_zip)
    return tile_dir

for city, cfg in TILE_CONFIG.items():
    print(f'\n{city}:')
    for z in cfg['tiles']:
        download_tile(z, BLDG_DIR)

print('\nAll tiles ready.')

Resolved 3/3 tile URLs from Figshare API

Chile:
  Downloaded 273.3 MB
  Extracting...
  Downloaded 79.1 MB
  Extracting...

Brazil:
  Downloaded 434.3 MB
  Extracting...

All tiles ready.


In [16]:
def compute_building_features(dataset_csv, building_dirs, city_bbox, buffer_m=100):
    """
    Compute 15 building morphology features per point within a square buffer.
    Loads shapefiles from downloaded 3D-GloBFP tile directories.
    """
    t0 = time.time()
    df = pd.read_csv(dataset_csv)

    # Load + merge all tile shapefiles
    gdfs = []
    for tile_dir in building_dirs:
        shp_files = [os.path.join(tile_dir, f) for f in os.listdir(tile_dir) if f.endswith('.shp')]
        for shp in shp_files:
            gdfs.append(gpd.read_file(shp))
    gdf_bld = pd.concat(gdfs, ignore_index=True)
    print(f'  Total buildings loaded: {len(gdf_bld)}')

    # Clip to city bbox
    w, s, e, n = city_bbox
    gdf_bld = gdf_bld[gdf_bld.geometry.intersects(box(w, s, e, n))].copy()
    print(f'  Buildings in city bbox: {len(gdf_bld)}')

    # Height values
    gdf_bld['height_val'] = pd.to_numeric(gdf_bld['Height'], errors='coerce').fillna(0)

    # Footprint areas in m2 (reproject to meters)
    gdf_bld_m = gdf_bld.to_crs(epsg=3857)
    gdf_bld['footprint_area'] = gdf_bld_m.geometry.area

    # KDTree on building centroids (lat/lon)
    bld_centroids = np.array([(g.centroid.x, g.centroid.y) for g in gdf_bld.geometry])
    bld_tree = cKDTree(bld_centroids)

    heights = gdf_bld['height_val'].values
    areas = gdf_bld['footprint_area'].values
    pt_coords = np.column_stack([df['Longitude'].values, df['Latitude'].values])
    buf_deg = buffer_m / 111000
    buffer_area_m2 = (2 * buffer_m) ** 2

    # Initialize output arrays
    n_pts = len(pt_coords)
    feats = {
        'building_density_100m': np.zeros(n_pts),
        'bldg_count_100m': np.zeros(n_pts),
        'bldg_mean_height': np.zeros(n_pts),
        'bldg_max_height': np.zeros(n_pts),
        'bldg_total_area': np.zeros(n_pts),
        'bldg_total_volume': np.zeros(n_pts),
        'bldg_height_std': np.zeros(n_pts),
        'bldg_height_range': np.zeros(n_pts),
        'bldg_height_max_ratio': np.zeros(n_pts),
        'sky_view_proxy': np.ones(n_pts),
        'volume_density': np.zeros(n_pts),
        'mean_bldg_volume': np.zeros(n_pts),
        'building_coverage_ratio': np.zeros(n_pts),
        'open_space_ratio': np.ones(n_pts),
        'compactness': np.zeros(n_pts),
    }

    for i in range(n_pts):
        nb = bld_tree.query_ball_point(pt_coords[i], r=buf_deg * 1.5)
        if not nb:
            continue

        h = heights[nb]
        a = areas[nb]
        n_bld = len(nb)
        h_mean = h.mean()
        h_max = h.max()
        total_a = a.sum()
        total_v = (h * a).sum()
        density = n_bld / buffer_area_m2

        feats['building_density_100m'][i] = density
        feats['bldg_count_100m'][i] = n_bld
        feats['bldg_mean_height'][i] = h_mean
        feats['bldg_max_height'][i] = h_max
        feats['bldg_total_area'][i] = total_a
        feats['bldg_total_volume'][i] = total_v
        feats['bldg_height_std'][i] = h.std() if n_bld > 1 else 0
        feats['bldg_height_range'][i] = h_max - h.min()
        feats['bldg_height_max_ratio'][i] = h_max / h_mean if h_mean > 0 else 0
        feats['sky_view_proxy'][i] = 1.0 / (1.0 + density * h_mean * 1e4)
        feats['volume_density'][i] = total_v / buffer_area_m2
        feats['mean_bldg_volume'][i] = total_v / n_bld
        coverage = min(total_a / buffer_area_m2, 1.0)
        feats['building_coverage_ratio'][i] = coverage
        feats['open_space_ratio'][i] = max(1.0 - coverage, 0.0)
        feats['compactness'][i] = density * h_mean

        if i % 5000 == 0 and i > 0:
            print(f'    {i}/{n_pts} points processed...')

    for col, vals in feats.items():
        df[col] = vals

    print(f'  Done in {time.time()-t0:.0f}s')
    return df

In [18]:
for city_name, cfg in TILE_CONFIG.items():
    print(f"\n{'='*60}")
    print(f"  {city_name} Building Features")
    print(f"{'='*60}")

    dataset_path = os.path.join(OUT_DIR, f'dataset_100m_{city_name}.csv')

    tile_dirs = []
    for z in cfg['tiles']:
        tile_dirs.append(os.path.join(BLDG_DIR, z.replace('.zip', '')))

    df_out = compute_building_features(dataset_path, tile_dirs, cfg['bbox'])

    out_path = os.path.join(OUT_DIR, f'buildings_{city_name}.csv')
    df_out.to_csv(out_path, index=False)
    print(f'  Saved: {out_path}')
    print(f'  Density: mean={df_out["building_density_100m"].mean():.6f}')
    print(f'  Height:  mean={df_out["bldg_mean_height"].mean():.1f}m, max={df_out["bldg_max_height"].max():.1f}m')


  Chile Building Features
  Total buildings loaded: 4591084
  Buildings in city bbox: 2954543
    5000/21662 points processed...
    10000/21662 points processed...
    15000/21662 points processed...
    20000/21662 points processed...
  Done in 86s
  Saved: C:\Users\micki\OneDrive\Desktop\Coding\BC-II\Final\buildings_Chile.csv
  Density: mean=0.004329
  Height:  mean=7.4m, max=75.6m

  Brazil Building Features
  Total buildings loaded: 5643491
  Buildings in city bbox: 2745688
    5000/28488 points processed...
    10000/28488 points processed...
    15000/28488 points processed...
    20000/28488 points processed...
    25000/28488 points processed...
  Done in 97s
  Saved: C:\Users\micki\OneDrive\Desktop\Coding\BC-II\Final\buildings_Brazil.csv
  Density: mean=0.004848
  Height:  mean=15.8m, max=96.3m


## Verification

Quick sanity checks on the extracted data. We confirm row counts match the competition CSVs, LST values are physically plausible (no cloud contamination), and building features have non-zero coverage for urban areas.

In [19]:
print('=== Output Verification ===\n')
for f in ['dataset_100m_Chile', 'dataset_100m_Brazil', 'dataset_100m_Sierra_Leone',
          'buildings_Chile', 'buildings_Brazil']:
    path = os.path.join(OUT_DIR, f'{f}.csv')
    if os.path.exists(path):
        df = pd.read_csv(path)
        lst_range = f'LST: {df["LST"].min():.1f} to {df["LST"].max():.1f}C' if 'LST' in df.columns else ''
        bldg_info = f'bldg cols: {sum(1 for c in df.columns if "bldg" in c or "building" in c)}' if 'building_density_100m' in df.columns else ''
        print(f'{f}: {df.shape[0]} rows x {df.shape[1]} cols | {lst_range} {bldg_info}')
    else:
        print(f'{f}: NOT FOUND')

=== Output Verification ===

dataset_100m_Chile: 21662 rows x 43 cols | LST: 32.4 to 51.9C 
dataset_100m_Brazil: 28488 rows x 43 cols | LST: 31.5 to 53.9C 
dataset_100m_Sierra_Leone: 14105 rows x 43 cols | LST: 31.4 to 60.1C 
buildings_Chile: 21662 rows x 58 cols | LST: 32.4 to 51.9C bldg cols: 11
buildings_Brazil: 28488 rows x 58 cols | LST: 31.5 to 53.9C bldg cols: 11
